## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, Jonathan Eugenio Gaeta

### To Do List:

1) download data from bezrealitky.cz - DONE and reality.idnes.cz, and from sreality.cz on other cities - DONE
2) create heatmap based on longitude and latitude - DONE
3) add property popups to map - DONE
4) download info on airbnb prices
5) conduct a simple analysis of rental price determinants

In [ ]:
# import packages
import json
import pandas as pd
import os
import requests 
import pandas as pd 
import time
import re 
import random 
import math
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import geopandas as gpd

### sreality.cz dataset

In [ ]:
# get sreality df from last request
df_sreality = pd.read_json("data/df_sreality.json")
df_sreality = pd.DataFrame(df_sreality)

In [ ]:
# or get newest sreality df, takes about 6 minutes
from function_scripts import request_sreality_all
df_sreality = request_sreality_all() 
df_sreality.to_json("data/df_sreality.json", orient="records")

In [ ]:
# get square meters (area) and flat type from name
from function_scripts import name_to_area
df_sreality['area'] = df_sreality.name.apply(name_to_area)
df_sreality['flat_type'] = df_sreality.name.apply(lambda x: x.split()[2])

# get link to listing and thumbnail image
from function_scripts import get_link_and_image
df_sreality = get_link_and_image(df_sreality)

display(df_sreality)

In [ ]:
# clean sreality dataset
df_sreality_clean = df_sreality[['locality', 'price', 'flat_type','area','gps','hash_id', 'url','image']].copy()

# get latitude and logitude, manually adjusted based on results
df_sreality_clean[['lat', 'lon']] = df_sreality_clean.gps.apply(lambda x: pd.Series({'lat': x['lat']+ 0.008, 'lon': x['lon']-0.008}))
df_sreality_clean = df_sreality_clean.drop(columns = ["gps", "hash_id"])

display(df_sreality_clean)


### bezrealitky.cz dataset

In [ ]:
# get bezrealitky df from last request
df_bezrealitky = pd.read_json("data/df_bezrealitky.json")
df_bezrealitky = pd.DataFrame(df_bezrealitky)

In [ ]:
# or get newest bezrealitky df, takes about 10 minutes
from function_scripts import request_bezrealitky
bezrealitky_url = "https://www.bezrealitky.cz/vyhledat?estateType=BYT&location=exact&offerType=PRONAJEM&osm_value=%C4%8Cesko&regionOsmIds=R51684&currency=CZK"
df_bezrealitky = request_bezrealitky(bezrealitky_url, 171)
df_bezrealitky.to_json("data/df_bezrealitky.json", orient="records")

In [ ]:
# convert flat_type to standard nomenclature
mapping = {
    'DISP_1_KK': '1+kk',
    'DISP_2_KK': '2+kk',
    'DISP_3_KK': '3+kk',
    'DISP_4_KK': '4+kk',
    'DISP_1_1': '1+1',
    'DISP_2_1': '2+1',
    'DISP_3_1': '3+1',
    'DISP_4_1': '4+1',
    'DISP_5_1': '5+1',
    'DISP_7_1': '7+1',
    'GARSONIERA': '1+kk',
    'OSTATNI': 'atypické',
    'UNDEFINED': 'atypické',
}

df_bezrealitky['flat_type'] = df_bezrealitky['flat_type'].map(mapping)
df_bezrealitky_clean = df_bezrealitky

display(df_bezrealitky_clean)

### Pooling datasets

In [ ]:
df_all = pd.concat([df_sreality_clean, df_bezrealitky_clean], ignore_index=True)

# remove non-czech properties (approximate)
df_all = df_all[
    (df_all['lat'] >= 48.5) &
    (df_all['lat'] <= 51.1) &
    (df_all['lon'] >= 12.0) &
    (df_all['lon'] <= 18.9)
]

display(df_all)
df_all.to_csv("data/df_all.csv")

In [ ]:
# datasets for map
df_heatmap = df_all[['lat', 'lon', 'price', 'area']].copy()
df_sreality_property = df_sreality_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()
df_bezrealitky_property = df_bezrealitky_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()

# save datasets
df_heatmap.to_parquet("data/df_heatmap.parquet")
df_sreality_property.to_parquet("data/df_sreality_property.parquet")
df_bezrealitky_property.to_parquet("data/df_bezrealitky_property.parquet")


### Heatmap

In [ ]:
from draw_heatmap import draw_heatmap
draw_heatmap(df_heatmap, df_sreality_property, df_bezrealitky_property)

### Analysis

In [ ]:
df_reg = df_all[['price', 'area', 'flat_type', 'locality']].dropna().copy()

# Extract district from locality
df_reg['district'] = df_reg['locality'].str.extract(r'^([^,]+)')

# Drop extreme outliers (top/bottom 1%)
q_low  = df_reg['price'].quantile(0.01)
q_high = df_reg['price'].quantile(0.99)
df_reg = df_reg[(df_reg['price'] > q_low) & (df_reg['price'] < q_high)]

print(df_reg.shape)
df_reg.head()

In [ ]:
# Keep only districts with enough observations
min_obs = 10
district_counts = df_reg['district'].value_counts()
df_reg = df_reg[df_reg['district'].isin(district_counts[district_counts >= min_obs].index)]

print(df_reg['district'].value_counts())

In [ ]:
# OLS: price ~ area + flat_type + district

model = smf.ols(
    formula='price ~ area + C(flat_type) + C(district)',
    data=df_reg
).fit()

print(model.summary())

In [ ]:
# Key diagnostics

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

#  residuals vs fitted
axes[0].scatter(model.fittedvalues, model.resid, alpha=0.3, s=10)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Fitted'); axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

#  price distribution
sns.histplot(df_reg['price'], bins=50, ax=axes[1])
axes[1].set_title('Price Distribution')

#  price vs area
axes[2].scatter(df_reg['area'], df_reg['price'], alpha=0.3, s=10)
axes[2].set_xlabel('Area (m²)'); axes[2].set_ylabel('Price (CZK)')
axes[2].set_title('Price vs Area')

plt.tight_layout()
plt.show()

In [ ]:
# Log-linear model
df_reg['log_price'] = np.log(df_reg['price'])

model_log = smf.ols(
    formula='log_price ~ area + C(flat_type) + C(district)',
    data=df_reg
).fit()

print(model_log.summary())

# Coefficient interpretation: area coef ≈ % change in price per extra m²
print(f"\nArea coefficient: {model_log.params['area']:.4f}")
print(f"→ Each extra m² is associated with ~{model_log.params['area']*100:.2f}% higher rent")

In [ ]:
#Quick summary table of district fixed effects
fe = model_log.params.filter(like='C(district)')
fe.index = fe.index.str.replace(r'C\(district\)\[T\.', '', regex=True).str.replace(']', '')
fe_sorted = fe.sort_values(ascending=False)

plt.figure(figsize=(10, max(4, len(fe_sorted)*0.3)))
fe_sorted.plot(kind='barh')
plt.axvline(0, color='red', lw=1)
plt.xlabel('Log-price premium vs baseline district')
plt.title('District Fixed Effects')
plt.tight_layout()
plt.show()